# IVI 데이터셋 진단 노트북

**목적**: `raw/IVI_social_isolation` 폴더의 모든 파일을 읽어 IVI 데이터셋 진단표 생성  
**주의**: 원자료는 절대 수정하지 않음 (읽기 전용)  
**출력**: `outputs/tables/ivi_dataset_diagnosis.csv`

---

## 목차

| # | 섹션 | 내용 |
|---|------|------|
| 1 | 환경 설정 | 라이브러리 임포트 / 경로 설정 |
| 2 | 공통 유틸리티 함수 | 파일 읽기 / 기본 정보 / 연도 탐지 / 결측·중복 확인 |
| 3 | IVI 전용 판단 함수 | 데이터셋 유형 / 공간단위 / 변환 등급 / 전처리 제안 |
| 4 | 진단 실행 함수 | 파일 1개 → 진단 행(들) 반환 |
| 5 | 전체 실행 | 진단표 생성 / 출력 / CSV 저장 |

---
## 1. 환경 설정

In [ ]:
# ─────────────────────────────────────────
# 1-1. 라이브러리 임포트
# ─────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', None)

print("라이브러리 임포트 완료")

In [ ]:
# ─────────────────────────────────────────
# 1-2. 경로 설정
# notebooks/ 에서 실행하므로 .parent 로 프로젝트 루트 이동
# config.py 에서 DATA_DIR, RAW_DIR, TABLE_DIR 을 가져옴
# ─────────────────────────────────────────
BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))

from config import DATA_DIR, RAW_DIR, TABLE_DIR

# 사용자 지정 경로 변수
folder_path = "raw/IVI_social_isolation"              # DATA_DIR 하위 상대 경로
output_path = "output/tables/ivi_dataset_diagnosis.csv"   # 참고용 표기

# 실제 경로 (Path 객체)
IVI_DIR    = RAW_DIR / "IVI_social_isolation"
OUTPUT_CSV = TABLE_DIR / "ivi_dataset_diagnosis.csv"

print(f"BASE_DIR   : {BASE_DIR}")
print(f"IVI 데이터 : {IVI_DIR}")
print(f"출력 경로  : {OUTPUT_CSV}")

# 발견된 파일 목록 미리 확인
ivi_files = sorted(p for p in IVI_DIR.glob('*') if p.is_file() and not p.name.startswith('.'))
print(f"\n발견된 파일 수: {len(ivi_files)}")
for f in ivi_files:
    print(f"  {f.name}")

---
## 2. 공통 유틸리티 함수

> 파일 읽기 / 기본 정보 / 연도 탐지 / 결측치 / 중복  
> CDI · RII 노트북에서도 그대로 재사용 예정

In [ ]:
# ─────────────────────────────────────────
# 2-1. 파일 읽기
# CSV 는 utf-8-sig → utf-8 → cp949 → euc-kr 순 시도
# Excel 은 시트별로 각각 읽어서 리스트로 반환
# 반환 형식: [(df, sheet_name, error_msg), ...]
# ─────────────────────────────────────────
def read_file_safely(file_path):
    fp  = Path(file_path)
    ext = fp.suffix.lower()

    # CSV 파일
    if ext == '.csv':
        for enc in ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']:
            try:
                df = pd.read_csv(fp, encoding=enc)
                return [(df, None, None)]
            except Exception:
                continue
        return [(None, None, "CSV 읽기 실패: 지원 인코딩 모두 실패")]

    # Excel 파일 (시트 여러 개도 처리)
    elif ext in ['.xlsx', '.xls']:
        try:
            xl = pd.ExcelFile(fp)
            results = []
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(fp, sheet_name=sheet)
                    results.append((df, sheet, None))
                except Exception as e:
                    results.append((None, sheet, f"시트 읽기 실패: {e}"))
            return results or [(None, None, "시트 없음")]
        except Exception as e:
            return [(None, None, f"Excel 읽기 실패: {e}")]

    # 지원하지 않는 형식
    return [(None, None, f"지원하지 않는 파일 형식: {ext}")]


print("read_file_safely 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 2-2. 기본 정보 추출
# 행·열 수, 컬럼 목록을 dict 로 반환
# ─────────────────────────────────────────
def get_basic_info(df):
    return {
        'n_rows'  : len(df),
        'n_cols'  : len(df.columns),
        'columns' : ' | '.join(df.columns.astype(str).tolist()),
    }


print("get_basic_info 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 2-3. 연도 정보 탐지
# 서울 Open Data 와이드 포맷에서는 연도가 컬럼명으로 있음 (예: 2019, 2020)
# 탐지 순서:
#   1) 연도 관련 컬럼명(연도, 기준년도 등)에서 값 추출
#   2) 컬럼명 자체가 4자리 연도인 경우
#   3) 1, 2 로 못 찾으면 첫 5개 컬럼 값에서 추가 탐색
# ─────────────────────────────────────────
def detect_available_years(df):
    year_kw = ['year', '연도', '년도', '기준년도', '기준연도',
               '기준년월', '년월', '기간', '시점', '기준년']
    found = set()

    # 1) 연도 관련 컬럼에서 값 추출
    # (?:19|20) 비캡처 그룹 사용 → re.findall 이 4자리 전체를 반환
    for col in df.columns:
        if any(k in str(col).lower() for k in year_kw):
            try:
                for v in df[col].dropna().astype(str).head(20):
                    found.update(re.findall(r'\b(?:19|20)\d{2}\b', v))
            except Exception:
                pass

    # 2) 컬럼명 자체가 연도 (서울 Open Data 와이드 포맷)
    for col in df.columns:
        if re.match(r'^(19|20)\d{2}$', str(col).strip()):
            found.add(str(col).strip())

    # 3) 아직 못 찾은 경우 첫 5컬럼 값에서 추가 탐색
    if not found:
        for col in df.columns[:5]:
            try:
                for v in df[col].dropna().astype(str).head(10):
                    found.update(re.findall(r'\b(?:19|20)\d{2}\b', v))
            except Exception:
                pass

    return ', '.join(sorted(found)) if found else 'unknown'


# ─────────────────────────────────────────
# 2-4. 시간 단위 추정
# 컬럼명 키워드로 year / month / day / unknown 구분
# ─────────────────────────────────────────
def detect_time_unit(df):
    cols_str = ' '.join(str(c) for c in df.columns)

    if any(k in cols_str for k in ['기준년월', '년월', '기준월']):
        return 'month'
    if any(k in cols_str for k in ['날짜', '일자', '기준일']):
        return 'day'
    if any(re.match(r'^(19|20)\d{2}$', str(c).strip()) for c in df.columns):
        return 'year'  # 컬럼명이 연도 숫자인 경우
    if any(k in cols_str for k in ['연도', '년도', '기준년도', '기준연도', 'year']):
        return 'year'
    return 'unknown'


print("detect_available_years / detect_time_unit 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 2-5. 결측치 요약
# 결측이 있는 컬럼만 "컬럼명:개수" 형식으로 반환
# 결측 없으면 'no_missing' 반환
# ─────────────────────────────────────────
def summarize_missing(df):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        return 'no_missing'
    return ' | '.join(f"{c}:{int(n)}" for c, n in missing.items())


# ─────────────────────────────────────────
# 2-6. 중복 행 수 / 요약행 탐지
# 요약행: 합계·소계·총계·계 등이 들어간 행
#   → 지역별 결합 전에 제거해야 하므로 미리 파악
# ─────────────────────────────────────────
def count_duplicate_rows(df):
    return int(df.duplicated().sum())


def count_summary_rows(df):
    summary_kw = {'합계', '소계', '총계', '계', '전체', 'total'}
    mask = pd.Series(False, index=df.index)

    # 문자열 컬럼에서만 탐지 (숫자 컬럼 무시)
    for col in df.select_dtypes(include='object').columns:
        try:
            mask |= df[col].astype(str).str.strip().isin(summary_kw)
        except Exception:
            pass

    n = int(mask.sum())
    return f"요약행 {n}개 추정" if n > 0 else "없음"


print("summarize_missing / count_duplicate_rows / count_summary_rows 정의 완료")

---
## 3. IVI 전용 판단 함수

> 서울 Open Data 와이드 포맷 특성 반영  
> CDI · RII 에서는 이 섹션만 해당 지표에 맞게 교체할 예정

In [ ]:
# ─────────────────────────────────────────
# 3-1. 데이터셋 유형 탐지
# 파일명 키워드로 IVI 데이터셋 유형을 추정
# 등록인구 파일은 '항목' 컬럼 값을 추가로 확인해
# 한국인·외국인이 같이 있는지 판단
# ─────────────────────────────────────────
def detect_ivi_dataset_type(file_name, df):
    fn = file_name

    if '독거노인' in fn:
        return '독거노인현황'

    if '1인가구' in fn or '일인가구' in fn:
        return '1인가구_연령별'

    if '등록인구' in fn:
        # 항목 컬럼 값에 '등록외국인' 이 있으면 혼합 파일로 구분
        if '항목' in df.columns:
            vals = df['항목'].astype(str).unique()
            if any('외국인' in v for v in vals):
                return '등록인구+등록외국인'
        return '등록인구_연령별_동별'

    if '노인여가' in fn or '복지시설' in fn:
        return '노인여가복지시설'

    if '장애' in fn:
        return '장애유형별_동별'

    return '기타'


print("detect_ivi_dataset_type 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 3-2. 공간 단위 탐지
# 서울 Open Data 컬럼명 패턴:
#   동별(N) 컬럼  → 구별+동별 (자치구 계층 포함)
#   자치구별(N) / 자치구(N) 컬럼 → 구별 전용
# 반환: (spatial_unit, has_dong, has_gu)
# ─────────────────────────────────────────
def detect_ivi_spatial_unit(df, file_name):
    fn   = file_name
    cols = [str(c) for c in df.columns]

    # 동 관련 컬럼 또는 파일명 키워드
    has_dong = (
        '동별' in fn
        or any('동별' in c for c in cols)
        or any(c.strip() in {'동', '행정동', '행정동명', '읍면동'} for c in cols)
    )

    # 구 관련 컬럼 또는 파일명 키워드
    has_gu = (
        '구별' in fn
        or any('자치구' in c or '자치구별' in c for c in cols)
        or any(c.strip() in {'구', '자치구', '시군구'} for c in cols)
    )

    # 동별 자료는 자치구 계층을 항상 포함하므로 has_gu 도 True
    if has_dong:
        has_gu = True

    if has_dong:
        return '구별+동별', True, True
    if has_gu:
        return '구별', False, True
    return 'unknown', False, False


print("detect_ivi_spatial_unit 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 3-3. 연령·성별·핵심 컬럼 탐지
# 서울 Open Data 와이드 포맷은 연령·성별 정보가
# 컬럼명이 아닌 서브헤더 행 값에 있을 수 있음
# → 컬럼명 + 파일명 + 첫 3행 값을 모두 확인
# ─────────────────────────────────────────
def detect_has_age(df, file_name):
    age_kw = ['연령', '나이', 'age']

    # 파일명에 연령 관련 키워드가 있으면 True
    if any(k in file_name for k in age_kw):
        return True

    # 컬럼명 확인
    if any(k in ' '.join(str(c) for c in df.columns) for k in age_kw):
        return True

    # 서브헤더 행 값 확인 (예: '65~69세', '70세 이상' 등)
    try:
        for i in range(min(3, len(df))):
            row_str = ' '.join(str(v) for v in df.iloc[i].values)
            if re.search(r'\d+세|~\d+세|세미만|세이상|연령|나이', row_str):
                return True
    except Exception:
        pass

    return False


def detect_has_gender(df, file_name):
    gender_kw = ['성별', '남자', '여자', '남성', '여성']

    if '성별' in file_name:
        return True

    if any(k in ' '.join(str(c) for c in df.columns) for k in gender_kw):
        return True

    try:
        for i in range(min(3, len(df))):
            row_str = ' '.join(str(v) for v in df.iloc[i].values)
            if any(k in row_str for k in gender_kw):
                return True
    except Exception:
        pass

    return False


def detect_key_columns(df):
    """IVI 관련 핵심 변수 후보 컬럼명·서브헤더 값 탐지."""
    key_kw = [
        '고령', '노인', '독거', '1인가구', '일인가구', '인구', '세대', '가구',
        '장애', '장애인', '시설', '종사자', '소계', '합계', '총계',
        '남자', '여자', '여성', '남성', '연령', '나이', '외국인', '한국인',
    ]
    candidates = []

    # 컬럼명에서 탐지
    for col in df.columns:
        if any(k in str(col) for k in key_kw):
            candidates.append(str(col))

    # 서브헤더 행 값에서 추가 탐지 (와이드 포맷 대응)
    try:
        for i in range(min(3, len(df))):
            for v in df.iloc[i].dropna().astype(str).unique():
                if any(k in v for k in key_kw) and v not in candidates:
                    candidates.append(v)
    except Exception:
        pass

    # 중복 제거 후 최대 25개
    seen, unique = set(), []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)

    return ' | '.join(unique[:25]) if unique else '해당 없음'


print("detect_has_age / detect_has_gender / detect_key_columns 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 3-4. 변환 등급 판단
# A: 동별 직접 사용 (행정동 단위 분석 가능)
# B: 구별 직접 사용 (구 단위 분석만 가능)
# C: 등록외국인 보조검토 (핵심 변수 제외)
# D: 변환 어려움 (공간 단위 불명확)
# E: 읽기 실패
# ─────────────────────────────────────────
def assign_ivi_conversion_grade(info):
    if info.get('error'):
        return 'E', '읽기 실패'

    dataset_type = info.get('dataset_type', '')

    # 등록외국인 전용 행은 C 등급
    if dataset_type == '등록외국인':
        return 'C', '등록외국인 보조검토'

    if info.get('has_dong'):
        return 'A', '동별 직접 사용'
    if info.get('has_gu'):
        return 'B', '구별 직접 사용'

    return 'D', '변환 어려움'


print("assign_ivi_conversion_grade 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 3-5. 사용 역할 판단
# 핵심후보: IVI 지수 산출에 직접 투입
# 보조후보: 보조 변수로 활용
# 제외검토: 핵심 변수에서는 빼고 별도 검토
# ─────────────────────────────────────────
def assign_ivi_use_role(info):
    dataset_type = info.get('dataset_type', '')

    if dataset_type == '등록외국인':
        return '제외검토'

    role_map = {
        '독거노인현황'        : '핵심후보',
        '1인가구_연령별'      : '핵심후보',
        '등록인구_연령별_동별': '핵심후보',
        '등록인구+등록외국인' : '핵심후보',  # 등록인구(한국인) 기준
        '노인여가복지시설'    : '보조후보',
        '장애유형별_동별'     : '보조후보',
    }
    return role_map.get(dataset_type, '보조후보')


print("assign_ivi_use_role 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 3-6. 전처리 방향 제안
# 데이터셋 유형에 따라 이후 전처리에서 할 작업을 미리 기록
# ─────────────────────────────────────────
def suggest_ivi_preprocess(info):
    dataset_type = info.get('dataset_type', '')

    suggestions = {
        '독거노인현황'        : '동별 독거노인 수 추출 후 고령인구 대비 비율 산출',
        '1인가구_연령별'      : '구별 고령 1인가구 수 추출 후 구 단위 보조변수로 사용',
        '등록인구_연령별_동별': '동별 65세 이상 인구(한국인) 합산 → 고령인구 비율 기준값 산출',
        '등록인구+등록외국인' : '동별 65세 이상 한국인 합산; 외국인 데이터는 핵심 변수 제외 검토',
        '등록외국인'          : '등록외국인은 IVI 핵심 변수 제외 검토; 보조 분석 시 별도 활용',
        '노인여가복지시설'    : '구별 시설수 합산; 시설이 많을수록 취약도 낮으므로 부족도로 반전 필요',
        '장애유형별_동별'     : '동별 장애인 인원 합산 후 전체 인구 대비 비율 산출',
    }
    return suggestions.get(dataset_type, '전처리 방향 미정 — 추가 확인 필요')


print("suggest_ivi_preprocess 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 3-7. 특이사항 메모 구성
# 읽기 오류, 와이드 포맷 주의, 연도 단일/미확인,
# 구별 한계, 외국인 혼재, 요약행 주의 등을 자동 기록
# ─────────────────────────────────────────
def build_ivi_notes(info):
    notes = []

    # 읽기 오류가 있으면 바로 반환
    if info.get('error'):
        return f"읽기 오류: {info['error']}"

    dataset_type    = info.get('dataset_type', '')
    has_dong        = info.get('has_dong', False)
    has_gu          = info.get('has_gu', False)
    available_years = info.get('available_years', 'unknown')
    summary_rows    = info.get('total_or_summary_rows', '없음')

    # 시트명이 있는 경우 (Excel 다중 시트)
    if info.get('sheet_name'):
        notes.append(f"시트명: {info['sheet_name']}")

    # 서울 Open Data 와이드 포맷 주의사항 (모든 파일 공통)
    notes.append('서울 Open Data 와이드 포맷 — 첫 1~3행은 서브헤더, 실제 데이터는 이후 행부터')

    # 연도 정보 관련
    if available_years == 'unknown':
        notes.append('연도 컬럼 미확인 — 추가 확인 필요')
    elif ',' not in available_years:
        notes.append(f'단일 연도 자료({available_years}) — 시계열 분석 제한적')

    # 구별 자료의 행정동 분석 한계
    if has_gu and not has_dong:
        notes.append('구별 자료 — 행정동 단위 분석에는 한계 있음')

    # 등록외국인 혼재 파일 주의
    if '등록외국인' in dataset_type and dataset_type != '등록외국인':
        notes.append('파일 내 항목 컬럼에 등록외국인 데이터 포함 — 핵심 IVI 변수에서는 제외 검토')

    # 요약행 처리 주의
    if summary_rows != '없음':
        notes.append(f"{summary_rows} — 결합 전 제거 또는 별도 처리 필요")

    # 장애유형 파일에서 동 컬럼 미확인 시 경고
    if dataset_type == '장애유형별_동별' and not has_dong:
        notes.append('파일명은 동별이나 동 관련 컬럼 미확인 — 공간 단위 재확인 필요')

    # 등록외국인 전용 행 추가 안내
    if dataset_type == '등록외국인':
        notes.append('등록인구 파일 내 항목=등록외국인 행에서 분리 추출 | 고령 외국인 거주자 보조 분석 가능')

    return ' | '.join(notes) if notes else '없음'


print("build_ivi_notes 정의 완료")

---
## 4. IVI 진단 실행 함수

> 파일 1개를 받아 위 함수들을 순서대로 호출하고  
> 진단 행(dict) 의 리스트를 반환

In [ ]:
# ─────────────────────────────────────────
# 4-1. 단일 파일 진단 함수
# 등록인구+등록외국인 파일은 외국인 전용 행을 추가로 생성
# (같은 파일이지만 진단표에서는 두 줄로 표현)
# ─────────────────────────────────────────
def diagnose_ivi_file(file_path):
    fp        = Path(file_path)
    file_name = fp.name
    file_ext  = fp.suffix.lower().lstrip('.')

    read_results = read_file_safely(fp)
    rows = []

    for df, sheet_name, error in read_results:

        # 공통 메타 초기화
        info = {
            'file_name' : file_name,
            'file_ext'  : file_ext,
            'sheet_name': sheet_name,
            'error'     : error,
        }

        # 읽기 성공 시: 각 탐지 함수 순서대로 실행
        if df is not None:
            info.update(get_basic_info(df))

            info['dataset_type']           = detect_ivi_dataset_type(file_name, df)
            spatial_unit, has_dong, has_gu = detect_ivi_spatial_unit(df, file_name)
            info['spatial_unit']           = spatial_unit
            info['has_dong']               = has_dong
            info['has_gu']                 = has_gu
            info['has_age']                = detect_has_age(df, file_name)
            info['has_gender']             = detect_has_gender(df, file_name)
            info['available_years']        = detect_available_years(df)
            info['time_unit']              = detect_time_unit(df)
            info['key_columns_candidate']  = detect_key_columns(df)
            info['total_or_summary_rows']  = count_summary_rows(df)
            info['missing_summary']        = summarize_missing(df)
            info['duplicate_rows']         = count_duplicate_rows(df)

        # 읽기 실패 시: 기본값으로 채움
        else:
            info.update({
                'n_rows': None, 'n_cols': None, 'columns': None,
                'dataset_type': '기타', 'spatial_unit': 'unknown',
                'has_dong': False, 'has_gu': False,
                'has_age': False, 'has_gender': False,
                'available_years': 'unknown', 'time_unit': 'unknown',
                'key_columns_candidate': '해당 없음',
                'total_or_summary_rows': '확인 불가',
                'missing_summary': '확인 불가',
                'duplicate_rows': None,
            })

        # IVI 전용 판단
        grade, method = assign_ivi_conversion_grade(info)
        use_role      = assign_ivi_use_role(info)
        preprocess    = suggest_ivi_preprocess(info)
        notes         = build_ivi_notes(info)

        # dataset_name: 시트가 있으면 시트명 포함
        dataset_name = info.get('dataset_type', '기타')
        if sheet_name:
            dataset_name = f"{dataset_name} ({sheet_name})"

        # 진단 행 조립 (24개 컬럼)
        row = {
            'category'              : 'IVI',
            'dataset_name'          : dataset_name,
            'raw_file_name'         : file_name,
            'file_path'             : str(fp),
            'file_type'             : file_ext,
            'n_rows'                : info.get('n_rows'),
            'n_cols'                : info.get('n_cols'),
            'columns'               : info.get('columns'),
            'available_years'       : info.get('available_years', 'unknown'),
            'time_unit'             : info.get('time_unit', 'unknown'),
            'spatial_unit'          : info.get('spatial_unit', 'unknown'),
            'has_dong'              : info.get('has_dong', False),
            'has_gu'                : info.get('has_gu', False),
            'has_age'               : info.get('has_age', False),
            'has_gender'            : info.get('has_gender', False),
            'key_columns_candidate' : info.get('key_columns_candidate', '해당 없음'),
            'total_or_summary_rows' : info.get('total_or_summary_rows', '없음'),
            'missing_summary'       : info.get('missing_summary', 'no_missing'),
            'duplicate_rows'        : info.get('duplicate_rows'),
            'conversion_grade'      : grade,
            'conversion_method'     : method,
            'use_role'              : use_role,
            'suggested_preprocess'  : preprocess,
            'notes'                 : notes,
        }
        rows.append(row)

        # 등록외국인 전용 행 추가
        # 같은 파일에서 외국인 데이터가 분리되므로 진단표에 별도 행 추가
        if info.get('dataset_type') == '등록인구+등록외국인' and df is not None:
            foreign_info = dict(info)
            foreign_info['dataset_type'] = '등록외국인'

            foreign_row = dict(row)
            foreign_row.update({
                'dataset_name'         : '등록외국인',
                'conversion_grade'     : 'C',
                'conversion_method'    : '등록외국인 보조검토',
                'use_role'             : '제외검토',
                'suggested_preprocess' : suggest_ivi_preprocess(foreign_info),
                'notes'                : build_ivi_notes(foreign_info),
            })
            rows.append(foreign_row)

    return rows


print("diagnose_ivi_file 정의 완료")

---
## 5. 전체 실행

In [ ]:
# ─────────────────────────────────────────
# 5-1. 파일 목록 순회 → 진단표 생성
# ─────────────────────────────────────────
all_rows = []

print(f"총 {len(ivi_files)}개 파일 처리 시작\n")
for fp in ivi_files:
    print(f"  처리 중: {fp.name}")
    all_rows.extend(diagnose_ivi_file(fp))

df_diag = pd.DataFrame(all_rows)
print(f"\n진단표 완성: {len(df_diag)}행 × {len(df_diag.columns)}열")

In [ ]:
# ─────────────────────────────────────────
# 5-2. 요약 뷰 출력
# 핵심 항목만 추려서 한눈에 파악
# ─────────────────────────────────────────
summary_cols = [
    'category', 'dataset_name',
    'n_rows', 'n_cols',
    'available_years', 'time_unit',
    'spatial_unit', 'has_dong', 'has_gu',
    'has_age', 'has_gender',
    'conversion_grade', 'conversion_method', 'use_role',
]

print("=" * 120)
print("[ IVI 데이터셋 진단 — 요약 뷰 ]")
print("=" * 120)
display(df_diag[summary_cols])

In [ ]:
# ─────────────────────────────────────────
# 5-3. 상세 뷰 출력
# 결측·중복·요약행 현황 및 전처리 방향·메모 확인
# ─────────────────────────────────────────
detail_cols = [
    'dataset_name',
    'key_columns_candidate',
    'total_or_summary_rows',
    'missing_summary',
    'duplicate_rows',
    'suggested_preprocess',
    'notes',
]

print("=" * 120)
print("[ IVI 데이터셋 진단 — 상세 뷰 ]")
print("=" * 120)
display(df_diag[detail_cols])

In [ ]:
# ─────────────────────────────────────────
# 5-4. 파일별 전체 컬럼명 목록 확인
# 전처리 단계에서 실제 컬럼명 참고용
# ─────────────────────────────────────────
print("=" * 120)
print("[ IVI 데이터셋 — 파일별 컬럼 목록 ]")
print("=" * 120)

for _, r in df_diag[['dataset_name', 'raw_file_name', 'columns']].iterrows():
    print(f"\n● {r['dataset_name']}  ({r['raw_file_name']})")
    print(f"  {r['columns']}")

In [ ]:
# ─────────────────────────────────────────
# 5-5. CSV 저장
# 폴더가 없으면 자동 생성 후 저장
# 한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용
# ─────────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_diag.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"저장 완료: {OUTPUT_CSV}")
print(f"\n── 진단표 구조 ──")
print(f"  행 수  : {len(df_diag)}")
print(f"  열 수  : {len(df_diag.columns)}")
print(f"  컬럼명 : {list(df_diag.columns)}")

---
## 6. CDI 데이터셋 진단 — 환경 설정

> **CDI(생활 인프라 사막 지수)** 에 사용할 데이터셋 진단
> 인코딩: cp949 (IVI 와 다름)
> 출력: `outputs/tables/cdi_dataset_diagnosis.csv`

| # | 섹션 | 내용 |
|---|------|------|
| 6 | 환경 설정 | CDI 경로 설정 |
| 7 | CDI 전용 판단 함수 | 데이터셋 유형 / 연도 / 플래그 / 공간단위 / 등급 / 메모 |
| 8 | 진단 실행 함수 | 파일 1개 → CDI 진단 행 반환 |
| 9 | 전체 실행 | CDI 진단표 생성 / 출력 / CSV 저장 |

In [ ]:
# ─────────────────────────────────────────
# 6-1. CDI 경로 설정
# ─────────────────────────────────────────
CDI_DIR        = RAW_DIR / "CDI_infra_desert"
CDI_OUTPUT_CSV = TABLE_DIR / "cdi_dataset_diagnosis.csv"

print(f"CDI 데이터 : {CDI_DIR}")
print(f"출력 경로  : {CDI_OUTPUT_CSV}")

cdi_files = sorted(p for p in CDI_DIR.glob('*') if p.is_file() and not p.name.startswith('.'))
print(f"\n발견된 파일 수: {len(cdi_files)}")
for f in cdi_files:
    print(f"  {f.name}")

---
## 7. CDI 전용 판단 함수

> 빅데이터캠퍼스 샘플 / 격자·행정동·상권 공간단위 차이 반영
> 공통 함수(Section 2)는 그대로 재사용

In [ ]:
# ─────────────────────────────────────────
# 7-1. CDI 데이터셋 유형 탐지
# 파일명 키워드 기반
# 상권분석서비스는 연도(2019/2020/2021)별로 별도 행 생성
# ─────────────────────────────────────────
def detect_cdi_dataset_type(file_name, df):
    fn = file_name

    if '소비지역별' in fn and '행정동' in fn:
        return '소비_일별_행정동'
    if '시간대별' in fn and '행정동' in fn:
        return '소비_일별_시간대별_행정동'
    if '내국인' in fn and '격자' in fn:
        return '소비_일별_시간대별_격자50'
    if '시간대별' in fn and '격자' in fn:
        return '소비_시간대별_격자250'
    if '상권분석서비스' in fn or '점포' in fn:
        m = re.search(r'(20\d{2})', fn)
        yr = m.group(1) if m else 'unknown'
        return f'상권분석서비스_점포_{yr}'
    if '약국' in fn:
        return '약국_운영시간_정보'
    return 'CDI_기타'


print("detect_cdi_dataset_type 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-2. CDI 연도·시간단위 탐지
# detect_available_years 보완:
#   - 2010~2030 범위 필터링으로 운영시간(1900, 1930 등) 오탐 방지
#   - 기준일자('20230101') → 앞 4자리로 연도 추출
#   - 기준_년분기_코드('20191') → 앞 4자리로 연도 추출
#   - 파일명에서 연도 추출
# ─────────────────────────────────────────
def detect_cdi_years(df, file_name):
    found = set()

    # 1) 공통 함수 결과를 합리적 연도 범위로 필터링
    common = detect_available_years(df)
    if common != 'unknown':
        for y in common.split(', '):
            if y.isdigit() and 2010 <= int(y) <= 2030:
                found.add(y)

    # 2) 날짜형 컬럼에서 앞 4자리 추출 (기준일자=20230101 등)
    date_kw = ['기준일자', '일자', '기준_년분기', '기준년분기']
    for col in df.columns:
        if any(k in str(col) for k in date_kw):
            try:
                for v in df[col].dropna().astype(str).head(5):
                    if len(v) >= 4 and v[:4].isdigit():
                        y = v[:4]
                        if 2000 <= int(y) <= 2030:
                            found.add(y)
            except Exception:
                pass

    # 3) 파일명에서 연도 추출
    for m in re.finditer(r'(20\d{2})', file_name):
        found.add(m.group(1))

    return ', '.join(sorted(found)) if found else 'unknown'


def detect_cdi_time_unit(df):
    cols_str = ' '.join(str(c) for c in df.columns)
    if '분기' in cols_str:
        return 'quarter'
    return detect_time_unit(df)


print("detect_cdi_years / detect_cdi_time_unit 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-3. CDI 플래그 탐지
# 빅데이터캠퍼스 샘플 / 공간 단위 / 데이터 속성 플래그
# ─────────────────────────────────────────
def detect_cdi_flags(file_name, df):
    fn   = file_name
    cols = ' '.join(str(c) for c in df.columns)

    return {
        'is_bigdata_campus_sample': '서울시민의' in fn or '내국인' in fn,
        'center_visit_required'   : '서울시민의' in fn or '내국인' in fn,
        'has_dong_code'      : any(k in cols for k in ['행정동코드', '고객행정동코드', '행정동_코드']),
        'has_grid'           : any(k in cols for k in ['격자_250', '격자_50', '격자250', '격자50', '격자']),
        'has_commercial_area': any(k in cols for k in ['상권_코드', '상권코드', '상권_구분', '상권_코드_명']),
        'has_address'        : any(k in cols for k in ['주소', '도로명주소', '지번주소', '소재지']),
        'has_coordinates'    : any(k in cols for k in ['경도', '위도', 'lon', 'lat', 'x좌표', 'y좌표',
                                                        '병원경도', '병원위도']),
        'has_time'           : any(k in cols for k in ['시간대', '운영시간', '진료시간']),
        'has_consumption'    : any(k in cols for k in ['카드이용', '소비', '이용금액', '이용건수',
                                                        '카드이용금액계', '카드이용건수계']),
        'has_business_type'  : any(k in cols for k in ['업종', '업종대분류', '서비스_업종_코드_명', '업태']),
    }


print("detect_cdi_flags 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-4. CDI 공간 단위 탐지
# 우선순위: 행정동코드 > 상권코드 > 격자 > 좌표 > 주소
# ─────────────────────────────────────────
def detect_cdi_spatial_unit(flags):
    if flags.get('has_dong_code'):
        return '행정동코드'
    if flags.get('has_commercial_area'):
        return '상권코드'
    if flags.get('has_grid'):
        return '격자'
    if flags.get('has_coordinates'):
        return '좌표(위경도)'
    if flags.get('has_address'):
        return '주소'
    return 'unknown'


print("detect_cdi_spatial_unit 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-5. CDI 핵심 컬럼 후보 탐지
# ─────────────────────────────────────────
def detect_key_columns_cdi(df):
    key_kw = [
        '카드이용금액', '카드이용건수', '이용금액', '이용건수',
        '점포_수', '점포수', '업종', '서비스_업종', '상권',
        '운영시간', '전화번호', '경도', '위도', '주소',
        '고객행정동', '행정동코드', '격자', '시간대',
    ]
    candidates = [str(c) for c in df.columns if any(k in str(c) for k in key_kw)]
    seen, unique = set(), []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    return ' | '.join(unique[:25]) if unique else '해당 없음'


print("detect_key_columns_cdi 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-6. CDI 변환 등급 판단
# A: 행정동코드 → 직접 매핑 가능
# B: 격자 매핑 후처리 필요 / 좌표 공간조인
# C: 상권-행정동 매핑 필요 / 지오코딩 필요
# D: 공간 단위 불명확
# E: 읽기 실패
# ─────────────────────────────────────────
def assign_cdi_conversion_grade(info):
    if info.get('error'):
        return 'E', '읽기 실패'

    dataset_type = info.get('dataset_type', '')
    spatial      = info.get('spatial_unit', 'unknown')

    # 데이터셋 유형별 우선 판단
    if '상권분석서비스' in dataset_type:
        return 'C', '상권-행정동 매핑 필요'
    if dataset_type == '약국_운영시간_정보':
        return 'B', '좌표 공간조인'

    if spatial == '행정동코드':
        return 'A', '행정동코드 직접 매핑'
    if spatial == '격자':
        return 'B', '행정동 매핑 후처리 필요'
    if spatial == '상권코드':
        return 'C', '상권-행정동 매핑 필요'
    if spatial in ['주소', '좌표(위경도)']:
        return 'C', '지오코딩 필요'
    return 'D', '공간 단위 불명확'


print("assign_cdi_conversion_grade 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-7. CDI 사용 역할 판단
# 핵심후보: CDI 지수 산출에 직접 투입 가능
# 센터방문필요_검증후보: 빅데이터캠퍼스 방문 후 등급/순위로 가공해 사용
# 센터방문필요_제외검토: 방문해도 핵심 변수 적용 어려움, 보조 검증 수준
# ─────────────────────────────────────────
def assign_cdi_use_role(info):
    dataset_type = info.get('dataset_type', '')

    role_map = {
        '소비_일별_행정동'          : '센터방문필요_검증후보',
        '소비_일별_시간대별_행정동' : '센터방문필요_검증후보',
        '소비_시간대별_격자250'     : '센터방문필요_검증후보',
        '소비_일별_시간대별_격자50' : '센터방문필요_제외검토',
        '약국_운영시간_정보'        : '핵심후보',
    }
    if dataset_type in role_map:
        return role_map[dataset_type]
    if '상권분석서비스' in dataset_type:
        return '핵심후보'
    return '보조후보'


print("assign_cdi_use_role 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-8. CDI 전처리 방향 제안
# ─────────────────────────────────────────
def suggest_cdi_preprocess(info):
    dataset_type = info.get('dataset_type', '')

    if '상권분석서비스' in dataset_type:
        return '서비스업종코드명에서 편의점·슈퍼·식료품·의료 관련 업종 필터링 후 상권-행정동 매핑 또는 행정동 집계'

    suggestions = {
        '소비_일별_행정동'          : '센터 방문 후 전체 데이터에서 고객행정동 코드 의미 확인, 원 소비금액/건수 대신 필수소비활성도 등급/순위로 산출',
        '소비_일별_시간대별_행정동' : '센터 방문 후 전체 데이터에서 시간대별 소비 원값이 아닌 행정동별 소비활성도 등급/순위로 산출',
        '소비_시간대별_격자250'     : '센터 방문 후 격자-행정동 매핑 가능성 확인, 원 격자값 반출 금지',
        '소비_일별_시간대별_격자50' : '격자 단위가 너무 세밀하므로 핵심 CDI 변수에서는 제외하고 필요 시 센터 내 보조 검증만 검토',
        '약국_운영시간_정보'        : '위도/경도를 이용해 행정동 경계와 공간조인 후 행정동별 약국 수 산출',
    }
    return suggestions.get(dataset_type, '전처리 방향 미정 — 추가 확인 필요')


print("suggest_cdi_preprocess 정의 완료")

In [ ]:
# ─────────────────────────────────────────
# 7-9. CDI 메모 구성
# export_risk_note: 개인정보·반출 위험 주의사항
# center_action_note: 빅데이터캠퍼스 방문 시 할 일
# build_cdi_notes: 특이사항 종합
# ─────────────────────────────────────────
def build_cdi_export_risk_note(info):
    dataset_type = info.get('dataset_type', '')

    if not info.get('is_bigdata_campus_sample'):
        return '공개 데이터 — 반출 제한 없음'

    # 소비_일별_행정동: 원값 반출 금지 명시
    if dataset_type == '소비_일별_행정동':
        return '원값 반출 주의: 센터 내 처리 후 등급/순위/플래그만 반출'

    notes = ['빅데이터캠퍼스 샘플 — 외부 반출 불가']
    if info.get('has_consumption'):
        notes.append('소비 금액·건수 집계 데이터 — 분석 결과물 반출 시 캠퍼스 심의 필요')
    if info.get('has_dong_code'):
        notes.append('행정동 단위 집계 — 개인 식별 위험 낮음')
    if info.get('has_grid'):
        notes.append('격자 단위 — 집계 해상도 확인 후 재식별 위험 검토')
    return ' | '.join(notes)


def build_cdi_center_action_note(info):
    if not info.get('is_bigdata_campus_sample'):
        return '해당 없음 (공개 데이터)'

    actions = ['빅데이터캠퍼스 방문 신청 후 원격 분석 환경에서 접근']
    if info.get('has_consumption'):
        actions.append('카드이용 데이터 신청 시 사용 목적 명시 (공모전 참가)')
    if info.get('has_grid'):
        actions.append('격자→행정동 매핑 테이블 캠퍼스 내 제공 여부 확인')
    actions.append('분석 결과물 반출 시 개인정보 심의 신청')
    return ' | '.join(actions)


def build_cdi_notes(info):
    if info.get('error'):
        return f"읽기 오류: {info['error']}"

    notes           = []
    dataset_type    = info.get('dataset_type', '')
    spatial         = info.get('spatial_unit', 'unknown')
    available_years = info.get('available_years', 'unknown')
    summary_rows    = info.get('total_or_summary_rows', '없음')

    if info.get('is_bigdata_campus_sample'):
        notes.append('빅데이터캠퍼스 샘플 — 방문 신청 후 원내 분석 환경에서만 사용 가능')

    notes.append('인코딩 cp949 — 읽기 시 encoding="cp949" 지정 필수')

    if spatial == '격자':
        notes.append('격자 단위 → 행정동 매핑 테이블 별도 확보 필요')
    if spatial == '상권코드':
        notes.append('상권 코드 → 서울시 상권 코드 매핑 테이블로 행정동 연결')
    if spatial in ['주소', '좌표(위경도)']:
        notes.append('주소/좌표 → 지오코딩 또는 역지오코딩으로 행정동 매핑 필요')
    if available_years == 'unknown':
        notes.append('연도 정보 미확인 — 기준일자 컬럼에서 수동 확인 필요')
    # 약국 파일 전용 추가 안내
    if dataset_type == '약국_운영시간_정보':
        notes.append('기준일자/영업상태/폐업 여부 컬럼 확인 후 현재 운영 중인 약국만 필터링 필요')
    if '상권분석서비스' in dataset_type:
        notes.append('2019/2020/2021년 파일 3개 — 분기별 자료; 연도별 병합 후 사용 권장')
    if summary_rows != '없음':
        notes.append(f"{summary_rows} — 결합 전 제거 또는 별도 처리 필요")

    return ' | '.join(notes) if notes else '없음'


print("build_cdi_export_risk_note / build_cdi_center_action_note / build_cdi_notes 정의 완료")

---
## 8. CDI 진단 실행 함수

> 파일 1개를 받아 CDI 전용 함수들을 순서대로 호출하고
> 진단 행(dict) 의 리스트를 반환
> IVI 24개 컬럼 + CDI 전용 12개 컬럼 = 총 36개 컬럼

In [ ]:
# ─────────────────────────────────────────
# 8-1. CDI 단일 파일 진단 함수
# IVI 와 달리:
#   - CDI 전용 연도 탐지(detect_cdi_years) 사용
#   - 플래그 탐지(detect_cdi_flags) 실행
#   - CDI 추가 컬럼 12개 포함
# ─────────────────────────────────────────
def diagnose_cdi_file(file_path):
    fp        = Path(file_path)
    file_name = fp.name
    file_ext  = fp.suffix.lower().lstrip('.')

    read_results = read_file_safely(fp)
    rows = []

    for df, sheet_name, error in read_results:
        info = {
            'file_name' : file_name,
            'file_ext'  : file_ext,
            'sheet_name': sheet_name,
            'error'     : error,
        }

        if df is not None:
            info.update(get_basic_info(df))

            info['dataset_type']          = detect_cdi_dataset_type(file_name, df)
            flags                         = detect_cdi_flags(file_name, df)
            info.update(flags)

            info['spatial_unit']          = detect_cdi_spatial_unit(flags)
            info['has_dong']              = flags.get('has_dong_code', False)
            info['has_gu']                = False
            info['has_age']               = False
            info['has_gender']            = False
            info['available_years']       = detect_cdi_years(df, file_name)
            info['time_unit']             = detect_cdi_time_unit(df)
            info['key_columns_candidate'] = detect_key_columns_cdi(df)
            info['total_or_summary_rows'] = count_summary_rows(df)
            info['missing_summary']       = summarize_missing(df)
            info['duplicate_rows']        = count_duplicate_rows(df)
        else:
            info.update({
                'n_rows': None, 'n_cols': None, 'columns': None,
                'dataset_type': 'CDI_기타', 'spatial_unit': 'unknown',
                'has_dong': False, 'has_gu': False,
                'has_age': False, 'has_gender': False,
                'is_bigdata_campus_sample': False, 'center_visit_required': False,
                'has_dong_code': False, 'has_grid': False,
                'has_commercial_area': False, 'has_address': False,
                'has_coordinates': False, 'has_time': False,
                'has_consumption': False, 'has_business_type': False,
                'available_years': 'unknown', 'time_unit': 'unknown',
                'key_columns_candidate': '해당 없음',
                'total_or_summary_rows': '확인 불가',
                'missing_summary': '확인 불가',
                'duplicate_rows': None,
            })

        grade, method = assign_cdi_conversion_grade(info)
        use_role      = assign_cdi_use_role(info)
        preprocess    = suggest_cdi_preprocess(info)
        export_risk   = build_cdi_export_risk_note(info)
        center_action = build_cdi_center_action_note(info)
        notes         = build_cdi_notes(info)

        dataset_name = info.get('dataset_type', 'CDI_기타')
        if sheet_name:
            dataset_name = f"{dataset_name} ({sheet_name})"

        row = {
            # IVI 공통 24개 컬럼
            'category'              : 'CDI',
            'dataset_name'          : dataset_name,
            'raw_file_name'         : file_name,
            'file_path'             : str(fp),
            'file_type'             : file_ext,
            'n_rows'                : info.get('n_rows'),
            'n_cols'                : info.get('n_cols'),
            'columns'               : info.get('columns'),
            'available_years'       : info.get('available_years', 'unknown'),
            'time_unit'             : info.get('time_unit', 'unknown'),
            'spatial_unit'          : info.get('spatial_unit', 'unknown'),
            'has_dong'              : info.get('has_dong', False),
            'has_gu'                : info.get('has_gu', False),
            'has_age'               : info.get('has_age', False),
            'has_gender'            : info.get('has_gender', False),
            'key_columns_candidate' : info.get('key_columns_candidate', '해당 없음'),
            'total_or_summary_rows' : info.get('total_or_summary_rows', '없음'),
            'missing_summary'       : info.get('missing_summary', 'no_missing'),
            'duplicate_rows'        : info.get('duplicate_rows'),
            'conversion_grade'      : grade,
            'conversion_method'     : method,
            'use_role'              : use_role,
            'suggested_preprocess'  : preprocess,
            'notes'                 : notes,
            # CDI 전용 추가 12개 컬럼
            'is_bigdata_campus_sample': info.get('is_bigdata_campus_sample', False),
            'center_visit_required'   : info.get('center_visit_required', False),
            'has_dong_code'           : info.get('has_dong_code', False),
            'has_grid'                : info.get('has_grid', False),
            'has_commercial_area'     : info.get('has_commercial_area', False),
            'has_address'             : info.get('has_address', False),
            'has_coordinates'         : info.get('has_coordinates', False),
            'has_time'                : info.get('has_time', False),
            'has_consumption'         : info.get('has_consumption', False),
            'has_business_type'       : info.get('has_business_type', False),
            'export_risk_note'        : export_risk,
            'center_action_note'      : center_action,
        }
        rows.append(row)

    return rows


print("diagnose_cdi_file 정의 완료")

---
## 9. CDI 전체 실행

In [ ]:
# ─────────────────────────────────────────
# 9-1. CDI 파일 목록 순회 → 진단표 생성
# ─────────────────────────────────────────
cdi_rows = []

print(f"총 {len(cdi_files)}개 파일 처리 시작\n")
for fp in cdi_files:
    print(f"  처리 중: {fp.name}")
    cdi_rows.extend(diagnose_cdi_file(fp))

df_cdi_diag = pd.DataFrame(cdi_rows)
print(f"\nCDI 진단표 완성: {len(df_cdi_diag)}행 × {len(df_cdi_diag.columns)}열")

In [ ]:
# ─────────────────────────────────────────
# 9-2. CDI 요약 뷰 출력
# ─────────────────────────────────────────
cdi_summary_cols = [
    'category', 'dataset_name',
    'n_rows', 'n_cols',
    'available_years', 'time_unit',
    'spatial_unit',
    'is_bigdata_campus_sample', 'center_visit_required',
    'has_dong_code', 'has_grid', 'has_commercial_area',
    'has_time', 'has_consumption', 'has_business_type',
    'conversion_grade', 'conversion_method', 'use_role',
]

print("=" * 140)
print("[ CDI 데이터셋 진단 — 요약 뷰 ]")
print("=" * 140)
display(df_cdi_diag[cdi_summary_cols])

In [ ]:
# ─────────────────────────────────────────
# 9-3. CDI 상세 뷰 출력
# ─────────────────────────────────────────
cdi_detail_cols = [
    'dataset_name',
    'key_columns_candidate',
    'total_or_summary_rows',
    'missing_summary',
    'duplicate_rows',
    'suggested_preprocess',
    'export_risk_note',
    'center_action_note',
    'notes',
]

print("=" * 140)
print("[ CDI 데이터셋 진단 — 상세 뷰 ]")
print("=" * 140)
display(df_cdi_diag[cdi_detail_cols])

In [ ]:
# ─────────────────────────────────────────
# 9-4. CDI 파일별 전체 컬럼명 목록
# ─────────────────────────────────────────
print("=" * 140)
print("[ CDI 데이터셋 — 파일별 컬럼 목록 ]")
print("=" * 140)

for _, r in df_cdi_diag[['dataset_name', 'raw_file_name', 'columns']].iterrows():
    print(f"\n● {r['dataset_name']}  ({r['raw_file_name']})")
    print(f"  {r['columns']}")

In [ ]:
# ─────────────────────────────────────────
# 9-5. CDI 진단표 CSV 저장
# ─────────────────────────────────────────
CDI_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_cdi_diag.to_csv(CDI_OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"저장 완료: {CDI_OUTPUT_CSV}")
print(f"\n── CDI 진단표 구조 ──")
print(f"  행 수  : {len(df_cdi_diag)}")
print(f"  열 수  : {len(df_cdi_diag.columns)}")
print(f"  컬럼명 : {list(df_cdi_diag.columns)}")